# Budget-Bounded Financial Research with AgentCore Payments and OpenAI

This notebook walks through a testnet research agent that can buy one approved x402-protected data source. The OpenAI model decides whether premium evidence is useful; AgentCore enforces the session budget and expiry outside the model.

> AgentCore Payments is in preview. This is an educational sample, not investment advice.

## 1. Install and configure

From the repository root:

```bash
python -m venv .venv
source .venv/bin/activate
pip install -e '.[dev]'
cp .env.example .env
```

Provision the Payment Manager, connector, and instrument with the official AgentCore Payments skill or AWS setup tutorial. Keep wallet-provider credentials out of this notebook and repository.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

load_dotenv(PROJECT_ROOT / ".env")

configuration = {
    "direct_openai_key": bool(os.getenv("OPENAI_API_KEY")),
    "bedrock_openai_enabled": os.getenv("BEDROCK_OPENAI_ENABLED", "false").lower() == "true",
    "payment_manager": bool(os.getenv("PAYMENT_MANAGER_ARN")),
    "payment_instrument": bool(os.getenv("PAYMENT_INSTRUMENT_ID")),
    "payment_session": bool(os.getenv("PAYMENT_SESSION_ID")),
    "payment_user": bool(os.getenv("PAYMENT_USER_ID")),
    "merchant_allowlist": bool(os.getenv("PAID_RESEARCH_ALLOWED_HOSTS")),
}
configuration

The check above prints only whether each setting exists. It does not display secrets or resource identifiers.

For Bedrock-hosted OpenAI models, set `BEDROCK_OPENAI_ENABLED=true`; the notebook obtains a short-lived bearer token from the active AWS credential chain.

Create a fresh per-run payment session from a terminal:

```bash
python scripts/create_payment_session.py --budget 0.25 --expiry-minutes 60
export PAYMENT_SESSION_ID=<printed-session-id>
```

The application creates this boundary before invoking the agent. The agent receives no tool that can increase the cap or extend the session.

## 2. Inspect the research request

The sample uses an AWS testnet endpoint. In a real finance workload, replace it with an approved x402 provider and review its data license.

In [ ]:
from paid_research.agent import build_prompt

query = "Assess the material near-term drivers and risks for AMZN."
paid_url = os.getenv(
    "PAID_RESEARCH_URL",
    "https://x402-test.genesisblock.ai/api/market-news",
)

print(build_prompt(query, paid_url))

## 3. Build the agent

The direct OpenAI path receives public web search, one approved paid fetch tool, and a redacted session-status tool. The Bedrock-hosted OpenAI path currently disables hosted web search because that endpoint rejects the `filters` field emitted by the Agents SDK. `parallel_tool_calls=False` keeps the purchase sequence easy to inspect.

In [ ]:
from typing import NoReturn

from paid_research.agent import build_agent
from paid_research.model_runtime import configure_model_runtime
from paid_research.x402 import X402PaymentClient

RUN_MODEL_LIVE = os.getenv("RUN_MODEL_LIVE", "false").lower() == "true"
RUN_PAYMENT_LIVE = os.getenv("RUN_PAYMENT_LIVE", "false").lower() == "true"


class DisabledPaymentClient:
    def fetch(self, _url: str) -> NoReturn:
        raise AssertionError("Model-only smoke test must not call the paid tool")

    def session_status(self) -> NoReturn:
        raise AssertionError("Model-only smoke test must not query payment state")


runtime = None
agent = None
if RUN_MODEL_LIVE or RUN_PAYMENT_LIVE:
    runtime = configure_model_runtime()
    payment_client = X402PaymentClient.from_env() if RUN_PAYMENT_LIVE else DisabledPaymentClient()
    agent = build_agent(
        payment_client,
        model=runtime.model,
        include_web_search=runtime.include_web_search,
    )
    print(
        f"Built {agent.name} with {runtime.provider}/{runtime.model}; "
        f"web search enabled: {runtime.include_web_search}"
    )
else:
    print("Set RUN_MODEL_LIVE=true for a model smoke test.")
    print("Set RUN_PAYMENT_LIVE=true only with a funded, delegated testnet wallet.")

## 4. Run the workflow

`RUN_MODEL_LIVE=true` runs a model-only Agents SDK smoke test without calling payment tools. `RUN_PAYMENT_LIVE=true` runs the paid research workflow and can spend testnet USDC.

In [ ]:
from agents import Runner

from paid_research.agent import run_research

if RUN_PAYMENT_LIVE:
    result = await run_research(query, paid_url=paid_url)
    print(result)
elif RUN_MODEL_LIVE:
    result = await Runner.run(
        agent,
        "Do not call tools. Reply with exactly PAID_RESEARCH_NOTEBOOK_OK.",
    )
    print(result.final_output)
else:
    print("Live run skipped.")

## 5. Prove the hard limit

Create another session with a budget below the endpoint price and rerun the same query:

```bash
python scripts/create_payment_session.py --budget 0.01 --expiry-minutes 15
```

The model can still ask for premium evidence. AgentCore rejects a payment that would exceed the available amount. The expected agent behavior is to report the missing evidence and narrow its conclusion, not to find another merchant or trial link.

## 6. Optional human approval

For a human checkpoint before each paid call, run the CLI with `--require-payment-approval`. The OpenAI Agents SDK pauses the run before the function executes and resumes the same state after approval or rejection.

```bash
paid-research "Assess AMZN" --paid-url "$PAID_RESEARCH_URL" --require-payment-approval
```

## 7. What to inspect

- OpenAI Traces: model calls, public search, paid tool choice, and final synthesis.
- AgentCore Payments telemetry: payment result, amount, remaining session budget, and signing latency.
- Final brief: claim-level citations, clear paid/public evidence labels, and a paid-data ledger.

For production, add role separation, AgentCore Gateway Policy, controlled network egress, data licensing controls, spend alarms, and evals for cost per successful research brief.